# Week 13. ML-блок для сквозного проекта

Проект: `variant_03`, погодные данные Open-Meteo по Новосибирску.

## Часть 0. Почему 99% качества может быть подозрительно

Модель нельзя проверять на данных, в которые она уже успела подсмотреть. В сломанном примере `y_target` попадает в признаки, а `StandardScaler` обучается на всем датасете до разделения на train/test. Из-за этого метрика может выглядеть высокой, но честной оценки нет.

Правило: target не кладем в `X`, все преобразования обучаем только на train, а для test используем только `transform`.

## Часть 1. Выбор сценария

В данных проекта нет целевой переменной, поэтому выбран вариант B — поиск аномалий. Это честнее, чем придумывать искусственный target для классификации.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
NORMALIZED_DIR = PROJECT_ROOT / 'data' / 'normalized' / 'variant_03'
OUTPUT_DIR = PROJECT_ROOT / 'docs' / 'ml'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_path = sorted(NORMALIZED_DIR.glob('*.csv'))[-1]
df = pd.read_csv(source_path, parse_dates=['time'])
df = df.sort_values('time').reset_index(drop=True)
df.head()

In [ ]:
metric_col = 'temperature'
q1 = df[metric_col].quantile(0.25)
q3 = df[metric_col].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
mean = df[metric_col].mean()
std = df[metric_col].std(ddof=0)

df_result = df.copy()
df_result['z_score'] = (df_result[metric_col] - mean) / std
df_result['is_anomaly'] = (df_result[metric_col] < lower_bound) | (df_result[metric_col] > upper_bound)
df_result['distance_from_iqr_bound'] = 0.0
lower_mask = df_result[metric_col] < lower_bound
upper_mask = df_result[metric_col] > upper_bound
df_result.loc[lower_mask, 'distance_from_iqr_bound'] = lower_bound - df_result.loc[lower_mask, metric_col]
df_result.loc[upper_mask, 'distance_from_iqr_bound'] = df_result.loc[upper_mask, metric_col] - upper_bound

anomalies = df_result[df_result['is_anomaly']].copy()
anomalies = anomalies.sort_values('distance_from_iqr_bound', ascending=False)
print(f'Q1: {q1:.2f}')
print(f'Q3: {q3:.2f}')
print(f'IQR: {iqr:.2f}')
print(f'Нижняя граница: {lower_bound:.2f}')
print(f'Верхняя граница: {upper_bound:.2f}')
print(f'Аномалий: {len(anomalies)} из {len(df_result)}')
anomalies[['time', 'temperature', 'z_score', 'distance_from_iqr_bound']]

In [ ]:
anomalies_out = anomalies[[
    'time', 'date', 'hour', 'temperature', 'z_score',
    'iqr_lower_bound', 'iqr_upper_bound', 'distance_from_iqr_bound'
]] if 'iqr_lower_bound' in anomalies.columns else anomalies.assign(
    iqr_lower_bound=lower_bound,
    iqr_upper_bound=upper_bound,
)[[
    'time', 'date', 'hour', 'temperature', 'z_score',
    'iqr_lower_bound', 'iqr_upper_bound', 'distance_from_iqr_bound'
]]
anomalies_out.to_csv(OUTPUT_DIR / 'anomalies_top.csv', index=False)

plot_path = OUTPUT_DIR / 'metrics.png'
plt.figure(figsize=(11, 5.5))
plt.plot(df_result['time'], df_result[metric_col], marker='o', linewidth=1.8, label='Температура')
plt.axhline(lower_bound, color='tab:red', linestyle='--', linewidth=1.2, label=f'IQR lower: {lower_bound:.2f}')
plt.axhline(upper_bound, color='tab:orange', linestyle='--', linewidth=1.2, label=f'IQR upper: {upper_bound:.2f}')
if not anomalies.empty:
    plt.scatter(anomalies['time'], anomalies[metric_col], color='tab:red', s=90, zorder=5, label='Аномалия')
plt.title('Week 13: поиск температурных аномалий')
plt.xlabel('Время')
plt.ylabel('Температура, C')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(plot_path, dpi=160)
plt.show()

## Вывод

Метод нашел один подозрительно холодный час: `2026-03-09 23:00:00`, температура `-4.1 C`. Это полезно как простой мониторинг погодных аномалий, но для надежной аналитики нужно больше исторических данных.